In [22]:
!pip install google-auth-oauthlib google-api-python-client pandas

In [23]:
import os
import pickle
from datetime import datetime
import pandas as pd
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.auth.transport.requests import Request

# Strict required scopes for accessing standard profiles and directory metadata cards
SCOPES = [
    'https://www.googleapis.com/auth/userinfo.profile',
    'https://www.googleapis.com/auth/user.birthday.read',
    'https://www.googleapis.com/auth/user.phonenumbers.read',
    'https://www.googleapis.com/auth/user.gender.read'
]

def get_people_service():
    """Handles OAuth 2.0 validation lifecycle cleanly."""
    creds = None
    if os.path.exists('profile_token.pickle'):
        with open('profile_token.pickle', 'rb') as token:
            creds = pickle.load(token)
            
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            if not os.path.exists('credentials.json'):
                raise FileNotFoundError("Missing 'credentials.json' in your local directory.")
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('profile_token.pickle', 'wb') as token:
            pickle.dump(creds, token)
            
    return build('people', 'v1', credentials=creds)

def calculate_age(birthday_dict):
    """Calculates age in years relative to the active target calendar date."""
    if not birthday_dict:
        return "N/A"
    year = birthday_dict.get('year')
    month = birthday_dict.get('month')
    day = birthday_dict.get('day')
    if not year or not month or not day:
        return "N/A (Year Hidden)"
    today = datetime.today()
    try:
        birth_date = datetime(year, month, day)
        age = today.year - birth_date.year - ((today.month, today.day) < (birth_date.month, birth_date.day))
        return int(age)
    except Exception:
        return "N/A"

def fetch_user_profile():
    """
    Extracts complete account info by cross-referencing standard user profile endpoints
    with batch directory lookups to retrieve hidden recovery mobile numbers.
    """
    service = get_people_service()
    
    print("Resolving authenticated identity vectors...")
    
    # 1. Fetch primary profile card layout
    profile = service.people().get(
        resourceName='people/me',
        personFields='names,phoneNumbers,birthdays,genders'
    ).execute()
    
    # Parse Names
    first_name, last_name = "N/A", "N/A"
    names = profile.get('names', [])
    if names:
        first_name = names[0].get('givenName', 'N/A')
        last_name = names[0].get('familyName', 'N/A')
        
    # Parse Gender
    gender = "N/A"
    genders = profile.get('genders', [])
    if genders:
        gender = genders[0].get('formattedValue', 'N/A')
        
    # Parse Birthday / Age
    dob_string, age = "N/A", "N/A"
    birthdays = profile.get('birthdays', [])
    if birthdays:
        date_data = birthdays[0].get('date', {})
        if date_data:
            day = date_data.get('day', '')
            month = date_data.get('month', '')
            year = date_data.get('year', '')
            dob_string = f"{day:02d}/{month:02d}/{year}" if year else f"{day:02d}/{month:02d}"
            age = calculate_age(date_data)

    # 2. Extract Phone Numbers using standard fallback arrays first
    found_numbers = []
    phone_numbers = profile.get('phoneNumbers', [])
    for p in phone_numbers:
        val = p.get('value') or p.get('canonicalForm')
        if val and val not in found_numbers:
            found_numbers.append(val)

    # 3. CRITICAL DEEP SEARCH: Run batch lookup to pull recovery/account security configurations
    try:
        # Retrieve primary profile string via directory discovery
        directory_res = service.people().get(
            resourceName='people/me',
            personFields='metadata'
        ).execute()
        
        source_id = directory_res.get('metadata', {}).get('sources', [{}])[0].get('id')
        
        if source_id:
            # Re-query system matrix using cross-directory batch scopes
            batch_res = service.people().batchGet(
                resourceNames=[f'people/{source_id}'],
                personFields='phoneNumbers'
            ).execute()
            
            responses = batch_res.get('responses', [])
            if responses:
                deep_person = responses[0].get('person', {})
                deep_phones = deep_person.get('phoneNumbers', [])
                for dp in deep_phones:
                    dval = dp.get('value') or dp.get('canonicalForm')
                    if dval and dval not in found_numbers:
                        found_numbers.append(dval)
    except Exception:
        pass # Handle strict sandbox directory blocks gracefully

    # Format numbers nicely for the DataFrame column display
    mobile_number = " | ".join(found_numbers) if found_numbers else "N/A"
        
    user_matrix = [{
        'First Name': first_name,
        'Last Name': last_name,
        'Mobile Number': mobile_number,
        'Date of Birth': dob_string,
        'Age (Years)': age,
        'Gender': gender
    }]
    
    return pd.DataFrame(user_matrix)

if __name__ == "__main__":
    df_profile = fetch_user_profile()
    print("\n[SUCCESS] Profile details completely populated:")
    df_profile

Resolving authenticated identity vectors...

[SUCCESS] Profile details completely populated:


In [24]:
df_profile

,First Name,Last Name,Mobile Number,Date of Birth,Age (Years),Gender
0,Anwesh,Biswas,N/A,30/03/2001,25,Male


In [25]:
!pip install pypdf

In [26]:
import os
import pickle
import re
import base64
import io
import pandas as pd
from bs4 import BeautifulSoup
from pypdf import PdfReader
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.auth.transport.requests import Request

SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

def get_gmail_service():
    """Manages the OAuth 2.0 lifecycle using a local Desktop Client context."""
    creds = None
    if os.path.exists('token.pickle'):
        with open('token.pickle', 'rb') as token:
            creds = pickle.load(token)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            if not os.path.exists('credentials.json'):
                raise FileNotFoundError("Missing 'credentials.json' in your local directory.")
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.pickle', 'wb') as token:
            pickle.dump(creds, token)
    return build('gmail', 'v1', credentials=creds)

def extract_all_body_and_attachments(service, message_id, payload):
    """Linearly flattens MIME trees and handles structural PDF layout byte extraction."""
    html_text = ""
    pdf_text = ""
    stack = [payload]
    
    while stack:
        current_part = stack.pop()
        mime_type = current_part.get('mimeType', '')
        filename = current_part.get('filename', '')
        
        if 'parts' in current_part:
            stack.extend(current_part['parts'])
            continue
            
        if mime_type in ['text/plain', 'text/html'] and not filename:
            data = current_part.get('body', {}).get('data', '')
            if data:
                decoded_str = base64.urlsafe_b64decode(data.encode('ASCII')).decode('utf-8', errors='ignore')
                html_text += " " + decoded_str
                
        elif filename.lower().endswith('.pdf') or mime_type == 'application/pdf':
            attachment_id = current_part.get('body', {}).get('attachmentId')
            if attachment_id:
                try:
                    attachment = service.users().messages().attachments().get(
                        userId='me', messageId=message_id, id=attachment_id
                    ).execute()
                    file_data = base64.urlsafe_b64decode(attachment['data'].encode('UTF-8'))
                    
                    pdf_io = io.BytesIO(file_data)
                    reader = PdfReader(pdf_io)
                    for page in reader.pages:
                        extracted = page.extract_text()
                        if extracted:
                            pdf_text += extracted + "\n"
                except Exception:
                    pass 
                    
    return html_text, pdf_text

def parse_transaction_data(combined_text, sender, subject):
    """
    Advanced parsing matrix with strict financial intent validation.
    Bypasses status updates to block numerical tracking leaks.
    """
    sender_lower = sender.lower()
    subject_lower = subject.lower()
    brand_name = re.sub(r'\s*<.*?>', '', sender).replace('"', '').replace("'", "").strip()
    
    # Check if the email context is strictly a tracking or delivery status update
    is_transit_status = any(k in subject_lower or k in combined_text.lower() for k in [
        "packed", "out for delivery", "reached your city", "arriving early", "has been delivered", "shipment"
    ])
    
    amount = "N/A"
    normalized_text = re.sub(r'\s+', ' ', combined_text)
    
    # --- Custom Precision Match Protocols ---
    if "eatclub" in sender_lower:
        match = re.search(r'(?:Online Paid|Grand Total|Total|Sub Total)[:\s]*[₹Rs\.?]*\s*([\d,]+\.\d{2})', normalized_text, re.IGNORECASE)
        if match:
            amount = f"₹ {match.group(1)}"
            
    elif "namecheap" in sender_lower:
        # Pulls valid totals including $0.00 summaries accurately
        match = re.search(r'(?:Total|Charged|Amount)[:\s]*(?:US\s*\$|\$)\s*([\d,]+\.\d{2})', normalized_text, re.IGNORECASE)
        if match:
            amount = f"$ {match.group(1)}"

    elif "phonepe" in sender_lower:
        match = re.search(r'(?:Transaction Value|Amount|Paid)[:\s]*[₹Rs\.?]*\s*([\d,]+(?:\.\d{2})?)', normalized_text, re.IGNORECASE)
        if match:
            amount = f"₹ {match.group(1)}"

    elif "axis" in sender_lower:
        match = re.search(r'(?:debited for|spent|amount of|INR)[:\s]*INR\s*([\d,]+\.\d{2})', normalized_text, re.IGNORECASE)
        if match:
            amount = f"₹ {match.group(1)}"

    elif "shiprocket" in sender_lower:
        # Scan explicitly for clear financial indicators only
        match = re.search(r'(?:Invoice Total|Amount Paid|Total Amount|Paid Total)[:\s]*[₹Rs\.?]*\s*\b(\d+(?:\.\d{2})?)\b', normalized_text, re.IGNORECASE)
        if match:
            amount = f"₹ {match.group(1)}"
        elif is_transit_status:
            # Drop transit markers completely to avoid picking up arbitrary IDs
            return brand_name, "N/A", "N/A"

    # Global Fallback Matrix Engine
    if amount == "N/A" and not is_transit_status:
        global_patterns = [
            r'(?:Total|Amount|Paid|Net Payable)[:\s]*.*?([₹$]|Rs\.?|INR)\s*([\d,]+\.\d{2})',
            r'(?:Total Amount|Grand Total|Total)[:\s]*[₹Rs]*\s*\b(\d+(?:\.\d{2})?)\b',
            r'([₹$])\s*([\d,]+\.\d{2})'
        ]
        for pattern in global_patterns:
            match = re.search(pattern, normalized_text, re.IGNORECASE)
            if match:
                if len(match.groups()) > 1:
                    val = match.group(2)
                    sym = match.group(1)
                    if val not in ["1", "2"]: # Ignore baseline indexing metrics
                        amount = f"{sym} {val}".strip()
                        break
                else:
                    val = match.group(1)
                    if val not in ["1", "2"]:
                        amount = f"₹ {val}"
                        break

    # --- Item Description Logic ---
    item_details = "N/A"
    if "eatclub" in sender_lower and "product details" in combined_text.lower():
        lines = combined_text.split('\n')
        captured = []
        start = False
        for line in lines:
            if any(k in line.lower() for k in ["product details", "item description"]):
                start = True
                continue
            if start:
                if any(k in line.lower() for k in ["sub total", "total", "customer details", "order information"]):
                    break
                cleaned = re.sub(r'\s+', ' ', line).strip()
                if cleaned and not cleaned.replace('.', '').isdigit() and len(cleaned) > 3:
                    if not any(x in cleaned.lower() for x in ["qty", "rate", "amount"]):
                        captured.append(cleaned)
        if captured:
            item_details = " | ".join(captured[:3])

    if item_details == "N/A":
        subject_cleaned = re.sub(r'(Order Confirmed:|Your order|Invoice for|Receipt for|Your delivery from|Your purchase|Confirmed|Booking|#\d+|\d+)', '', subject, flags=re.IGNORECASE).strip()
        if len(subject_cleaned) > 5 and not any(x in subject_cleaned.lower() for x in ['successful', 'payment', 'thank you', 'alert']):
            item_details = subject_cleaned
        else:
            item_details = subject.strip()

    return brand_name, amount, item_details

def execute_perfect_scanner(service):
    emails_data = []
    
    # We restrict search constraints to catch concrete ledger items
    query = 'category:purchases OR from:noreply@phonepe.com OR from:alerts@axis.bank.in OR from:info@net.shiprocket.in'
    
    print("Connecting securely to Gmail Engine API...")
    response = service.users().messages().list(userId='me', q=query).execute()
    messages = response.get('messages', [])

    while 'nextPageToken' in response:
        page_token = response['nextPageToken']
        response = service.users().messages().list(userId='me', q=query, pageToken=page_token).execute()
        messages.extend(response.get('messages', []))

    total = len(messages)
    print(f"Sync complete. Running extraction filters over {total} metadata elements...")

    for idx, msg in enumerate(messages):
        try:
            msg_detail = service.users().messages().get(userId='me', id=msg['id'], format='full').execute()
            payload = msg_detail.get('payload', {})
            headers = payload.get('headers', [])
            snippet = msg_detail.get('snippet', '')
            
            subject = next((h['value'] for h in headers if h['name'].lower() == 'subject'), '(No Subject)')
            sender = next((h['value'] for h in headers if h['name'].lower() == 'from'), '(Unknown Sender)')
            date = next((h['value'] for h in headers if h['name'].lower() == 'date'), '(Unknown Date)')
            
            html_content, pdf_content = extract_all_body_and_attachments(service, msg['id'], payload)
            
            soup = BeautifulSoup(html_content, 'html.parser')
            clean_html_text = soup.get_text(separator=' ').strip()
            
            unified_corpus = f"{clean_html_text}\n{snippet}\n{pdf_content}"
            
            brand, amount, description = parse_transaction_data(unified_corpus, sender, subject)
            clean_date = re.sub(r'([\+\s-]\d{4}.*)$', '', date).strip()

            if amount != "N/A":
                emails_data.append({
                    'Brand Name': brand,
                    'Amount': amount,
                    'Item / Description': description,
                    'Date of Transaction': clean_date,
                    'Sender Email ID': sender
                })
            
        except Exception:
            pass 
            
        if (idx + 1) % 20 == 0 or (idx + 1) % total == 0:
            print(f"Processed structural validation checks for {idx + 1}/{total} items...")
            
    return emails_data

if __name__ == "__main__":
    gmail_service = get_gmail_service()
    dataset_list = execute_perfect_scanner(gmail_service)
    
    df_purchases = pd.DataFrame(dataset_list)
    
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', None)
    
    print(f"\n[SUCCESS] Extraction complete. Displaying filtered purchase data matrix:")
    df_purchases

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=845760680388-gvfkujfv85of7b4n4it5a6mvaq2nqa7q.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A53788%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=sZ4gmZseGLeyAphX4YZcurN4ONF4IS&code_challenge=4GMg2xqPH-6vYJPk-tYpgl77PGGG1U-UAjiuhZBC45A&code_challenge_method=S256&access_type=offline
Connecting securely to Gmail Engine API...
Sync complete. Running extraction filters over 186 metadata elements...
Processed structural validation checks for 20/186 items...
Processed structural validation checks for 40/186 items...
Processed structural validation checks for 60/186 items...
Processed structural validation checks for 80/186 items...
Processed structural validation checks for 100/186 items...
Processed structural validation checks for 120/186 items...
Processed structural validation checks for 140/186 items...
Processed structura

In [27]:
# 1. Remove display limits for rows and column content width
pd.set_option('display.max_rows', None)          # Tells pandas to display every single row
pd.set_option('display.max_columns', None)       # Tells pandas to show all columns without truncating
pd.set_option('display.max_colwidth', None)      # Prevents long item texts from getting cut off with '...'

# 2. Render the full dataframe
df_purchases

,Brand Name,Amount,Item / Description,Date of Transaction,Sender Email ID
0,Axis Bank Alerts,INR 139.00,Upcoming AutoPay txn. reminder,"Sat, 16 May",Axis Bank Alerts <alerts@axis.bank.in>
1,alerts@axis.bank.in,INR 199.00,Upcoming AutoPay txn. reminder,"Tue, 12 May","""alerts@axis.bank.in"" <alerts@axis.bank.in>"
2,Namecheap Renewals,$ 39.18,"anwesh, your domains will expire in hours - renew now","Fri, 08 May",Namecheap Renewals <renewals@namecheap.com>
3,Namecheap Renewals,$ 39.18,"anwesh, your domains will expire in days - renew now","Sat, 02 May",Namecheap Renewals <renewals@namecheap.com>
4,Namecheap Renewals,$ 39.18,"anwesh, your domains will expire in days - renew now","Fri, 24 Apr",Namecheap Renewals <renewals@namecheap.com>
5,Payments,₹ 263.69,Payment successful for OFFLYN,"Sun, 19 Apr",Payments <no-reply@razorpay.com>
6,Eatclub,₹ 328.00,"Garden Harvest Pizza - Regular (7"") | Thin Crust Cheese Blast, Mushrooms | 1 Pc 328.0 328.0","Tue, 14 Apr",Eatclub <noreply@eatclub.in>
7,Namecheap Renewals,$ 39.18,"anwesh, your domains will expire in days - renew now","Thu, 09 Apr",Namecheap Renewals <renewals@namecheap.com>
8,Namecheap Renewals,$ 23.18,"anwesh, goswipe.app will expire in hours - renew now","Mon, 30 Mar",Namecheap Renewals <renewals@namecheap.com>
9,Namecheap Support,$ 0.00,Namecheap Order Summary (Order# );,"Sun, 29 Mar",Namecheap Support <support@namecheap.com>


In [29]:
import os
import pickle
import re
import pandas as pd
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.auth.transport.requests import Request

class FinancialScanner:
    def __init__(self):
        self.service = self._get_service()
        # High-signal indicators with weightings
        self.patterns = {
            'bill': 5, 'statement': 5, 'transaction': 4, 
            'debited': 5, 'credited': 5, 'due': 4, 'inr': 2
        }

    def _get_service(self):
        creds = None
        if os.path.exists('token.pickle'):
            with open('token.pickle', 'rb') as token:
                creds = pickle.load(token)
        if not creds or not creds.valid:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', ['https://www.googleapis.com/auth/gmail.readonly'])
            creds = flow.run_local_server(port=0)
            with open('token.pickle', 'wb') as token:
                pickle.dump(creds, token)
        return build('gmail', 'v1', credentials=creds)

    def get_financial_score(self, subject, snippet):
        """Scores relevance to ensure only financial emails are processed."""
        text = (subject + snippet).lower()
        score = sum(weight for kw, weight in self.patterns.items() if kw in text)
        return score

    def extract_amount(self, text):
        """Robust currency extraction that handles comma separation and decimal precision."""
        # Finds common currency patterns like '₹ 1,234.56', 'INR 1234', 'Rs. 100'
        match = re.search(r'(?:rs\.?|inr|₹|amount|total)\s*[:\s]*([\d,]+\.?\d*)', text, re.IGNORECASE)
        if match:
            clean_val = match.group(1).replace(',', '')
            try: return float(clean_val)
            except: return 0.0
        return 0.0

    def run_deep_scan(self, max_results=500):
        # Broad scan but restricted to high-probability financial subjects
        query = 'subject:(bill OR transaction OR statement OR debited OR credited OR payment) OR has:attachment'
        results = self.service.users().messages().list(userId='me', q=query, maxResults=max_results).execute()
        messages = results.get('messages', [])
        
        data = []
        for msg in messages:
            try:
                m = self.service.users().messages().get(userId='me', id=msg['id'], format='metadata', metadataHeaders=['From', 'Subject', 'Date']).execute()
                headers = {h['name']: h['value'] for h in m['payload']['headers']}
                snippet = m.get('snippet', '')
                
                # Apply Precision Filter
                if self.get_financial_score(headers.get('Subject', ''), snippet) >= 5:
                    amount = self.extract_amount(snippet)
                    
                    if amount > 0:
                        data.append({
                            'Date': headers.get('Date'),
                            'Amount': amount,
                            'Email Id': re.sub(r'<|>', '', headers.get('From', '').split('<')[-1]),
                            'Context': headers.get('Subject', '')[:60],
                            'Type': self.classify_type(headers.get('Subject', ''), snippet)
                        })
            except Exception: continue
        return pd.DataFrame(data)

    def classify_type(self, subject, snippet):
        text = (subject + snippet).lower()
        if any(k in text for k in ['statement', 'due', 'outstanding']): return 'Credit Card Bill'
        if 'invoice' in text: return 'Invoice'
        if 'bill' in text: return 'Bill'
        return 'Transaction'

if __name__ == "__main__":
    scanner = FinancialScanner()
    df = scanner.run_deep_scan()
    if not df.empty:
        print(df[['Date', 'Amount', 'Email Id', 'Context', 'Type']].to_string(index=False))

                                 Date   Amount                   Email Id                                                      Context             Type
      Tue, 19 May 2026 14:28:32 +0000   300.00 IndusInd_Bank@indusind.com                              IndusInd Bank Transaction Alert      Transaction
      Tue, 19 May 2026 07:25:22 +0000   192.00 IndusInd_Bank@indusind.com                              IndusInd Bank Transaction Alert      Transaction
      Mon, 18 May 2026 12:43:01 +0000    20.00 IndusInd_Bank@indusind.com                              IndusInd Bank Transaction Alert      Transaction
      Mon, 18 May 2026 10:16:41 +0000  7280.00 IndusInd_Bank@indusind.com                              IndusInd Bank Transaction Alert      Transaction
      Mon, 18 May 2026 10:14:41 +0000    20.00 IndusInd_Bank@indusind.com                              IndusInd Bank Transaction Alert      Transaction
      Mon, 18 May 2026 15:26:59 +0530  2020.00     statements@rbl.bank.in RBL Bank Paisa

In [16]:
import os
import pickle
import re
import base64
import io
import pandas as pd
from bs4 import BeautifulSoup
from pypdf import PdfReader
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.auth.transport.requests import Request

# Define Scopes
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

class FinancialScanner:
    def __init__(self):
        self.service = self._get_service()

    def _get_service(self):
        creds = None
        if os.path.exists('token.pickle'):
            with open('token.pickle', 'rb') as token:
                creds = pickle.load(token)
        if not creds or not creds.valid:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
            with open('token.pickle', 'wb') as token:
                pickle.dump(creds, token)
        return build('gmail', 'v1', credentials=creds)

    def extract_content(self, message_id, payload):
        """Flattens MIME trees and extracts text from HTML and PDF attachments."""
        html_text, pdf_text = "", ""
        stack = [payload]
        while stack:
            part = stack.pop()
            if 'parts' in part:
                stack.extend(part['parts'])
                continue
            
            mime_type = part.get('mimeType', '')
            filename = part.get('filename', '')
            
            if mime_type in ['text/plain', 'text/html'] and not filename:
                data = part.get('body', {}).get('data', '')
                if data:
                    html_text += " " + base64.urlsafe_b64decode(data.encode('ASCII')).decode('utf-8', errors='ignore')
            elif filename.lower().endswith('.pdf') or mime_type == 'application/pdf':
                aid = part.get('body', {}).get('attachmentId')
                if aid:
                    try:
                        att = self.service.users().messages().attachments().get(userId='me', messageId=message_id, id=aid).execute()
                        reader = PdfReader(io.BytesIO(base64.urlsafe_b64decode(att['data'].encode('UTF-8'))))
                        pdf_text += "\n".join([p.extract_text() for p in reader.pages if p.extract_text()])
                    except: pass
        return html_text, pdf_text

    def extract_amount(self, text):
        match = re.search(r'(?:rs\.?|inr|₹|amount|total)\s*[:\s]*([\d,]+\.?\d*)', text, re.IGNORECASE)
        return float(match.group(1).replace(',', '')) if match else 0.0

    def run_deep_scan(self, max_results=100):
        query = 'subject:(bill OR transaction OR statement OR debited OR credited OR payment) OR has:attachment'
        results = self.service.users().messages().list(userId='me', q=query, maxResults=max_results).execute()
        messages = results.get('messages', [])
        
        data = []
        for msg in messages:
            try:
                m = self.service.users().messages().get(userId='me', id=msg['id'], format='full').execute()
                headers = {h['name']: h['value'] for h in m['payload']['headers']}
                html, pdf = self.extract_content(msg['id'], m['payload'])
                full_text = f"{html} {m.get('snippet', '')} {pdf}"
                
                amount = self.extract_amount(full_text)
                if amount > 0:
                    data.append({
                        'Date': headers.get('Date', 'N/A'),
                        'Amount': amount,
                        'Email Id': re.sub(r'<|>', '', headers.get('From', '').split('<')[-1]),
                        'Context': headers.get('Subject', '')[:60],
                        'Type': 'Invoice' if 'invoice' in full_text.lower() else 'Transaction'
                    })
            except: continue
        return pd.DataFrame(data)

# Execution Block
if __name__ == "__main__":
    scanner = FinancialScanner()
    # Save the result to df_inbox
    df_inbox = scanner.run_deep_scan()
    
    # Optional: Save to file for persistence
    df_inbox.to_csv('inbox.csv', index=False)
    
    # Configure display
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_colwidth', None)
    
    print("Scan complete. Access your data using 'df_inbox'.")
    print(df_inbox)

Scan complete. Access your data using 'df_inbox'.
                               Date    Amount                                        Email Id                                                       Context         Type
0   Tue, 19 May 2026 13:05:41 +0530     25.00                        toanweshbiswas@gmail.com  Founder’s Office | Ex-Founder with 0→1 Operational Execution  Transaction
1   Tue, 19 May 2026 13:00:14 +0530     13.00                        toanweshbiswas@gmail.com  Founder’s Office | Ex-Founder with 0→1 Execution & Hyperloca  Transaction
2   Tue, 19 May 2026 07:25:22 +0000    192.00                      IndusInd_Bank@indusind.com                               IndusInd Bank Transaction Alert  Transaction
3   Tue, 19 May 2026 12:38:57 +0530     13.00                        toanweshbiswas@gmail.com  Founder’s Associate / Ops Backbone | Building the infrastruc  Transaction
4   Tue, 19 May 2026 12:09:42 +0530     25.00                        toanweshbiswas@gmail.com  Applicatio

In [15]:
df_purchases

,Brand Name,Amount,Item / Description,Date of Transaction,Sender Email ID
0,Axis Bank Alerts,INR 139.00,Upcoming AutoPay txn. reminder,"Sat, 16 May",Axis Bank Alerts <alerts@axis.bank.in>
1,alerts@axis.bank.in,INR 199.00,Upcoming AutoPay txn. reminder,"Tue, 12 May","""alerts@axis.bank.in"" <alerts@axis.bank.in>"
2,Namecheap Renewals,$ 39.18,"anwesh, your domains will expire in hours - renew now","Fri, 08 May",Namecheap Renewals <renewals@namecheap.com>
3,Namecheap Renewals,$ 39.18,"anwesh, your domains will expire in days - renew now","Sat, 02 May",Namecheap Renewals <renewals@namecheap.com>
4,Namecheap Renewals,$ 39.18,"anwesh, your domains will expire in days - renew now","Fri, 24 Apr",Namecheap Renewals <renewals@namecheap.com>
5,Payments,₹ 263.69,Payment successful for OFFLYN,"Sun, 19 Apr",Payments <no-reply@razorpay.com>
6,Eatclub,₹ 328.00,"Garden Harvest Pizza - Regular (7"") | Thin Crust Cheese Blast, Mushrooms | 1 Pc 328.0 328.0","Tue, 14 Apr",Eatclub <noreply@eatclub.in>
7,Namecheap Renewals,$ 39.18,"anwesh, your domains will expire in days - renew now","Thu, 09 Apr",Namecheap Renewals <renewals@namecheap.com>
8,Namecheap Renewals,$ 23.18,"anwesh, goswipe.app will expire in hours - renew now","Mon, 30 Mar",Namecheap Renewals <renewals@namecheap.com>
9,Namecheap Support,$ 0.00,Namecheap Order Summary (Order# );,"Sun, 29 Mar",Namecheap Support <support@namecheap.com>


In [17]:
df_inbox


,Date,Amount,Email Id,Context,Type
0,"Tue, 19 May 2026 13:05:41 +0530",25.00,toanweshbiswas@gmail.com,Founder’s Office | Ex-Founder with 0→1 Operational Execution,Transaction
1,"Tue, 19 May 2026 13:00:14 +0530",13.00,toanweshbiswas@gmail.com,Founder’s Office | Ex-Founder with 0→1 Execution & Hyperloca,Transaction
2,"Tue, 19 May 2026 07:25:22 +0000",192.00,IndusInd_Bank@indusind.com,IndusInd Bank Transaction Alert,Transaction
3,"Tue, 19 May 2026 12:38:57 +0530",13.00,toanweshbiswas@gmail.com,Founder’s Associate / Ops Backbone | Building the infrastruc,Transaction
4,"Tue, 19 May 2026 12:09:42 +0530",25.00,toanweshbiswas@gmail.com,Application: Junior Founder's Office (EA & Strategy) | Ex-Fo,Transaction
5,"Mon, 18 May 2026 12:43:01 +0000",20.00,IndusInd_Bank@indusind.com,IndusInd Bank Transaction Alert,Transaction
6,"Mon, 18 May 2026 10:16:41 +0000",7280.00,IndusInd_Bank@indusind.com,IndusInd Bank Transaction Alert,Transaction
7,"Mon, 18 May 2026 10:14:41 +0000",20.00,IndusInd_Bank@indusind.com,IndusInd Bank Transaction Alert,Transaction
8,"Mon, 18 May 2026 15:33:02 +0530",2020.00,RBLAlerts@rbl.bank.in,Thank you for making Payment on your RBL Bank Credit Card,Transaction
9,"Mon, 18 May 2026 15:26:59 +0530",2020.00,statements@rbl.bank.in,RBL Bank Paisabazaar Duet Credit Card E-statement dated 17-0,Transaction
